# TFiLM-Conditioned Stateful GR LSTM — Discretized Bins + Masked Huber/Delta Loss

**Google Colab**: Runtime → **GPU**. Open via *File → Open notebook → GitHub* (`5aola/Virtual-Analogue-Compressor-Modelling`); cell 1 clones the repo and mounts Drive for the dataset. **Push local changes before running.**

Successor of [`train_lstm_tfilm_gr.ipynb`](train_lstm_tfilm_gr.ipynb) (kept untouched for tracking). Three changes vs the scalar-sigmoid run:

1. **Discretized output head** (back to the 03 CREPE recipe): **71 bins @ 0.5 dB** over the measured **[−30, +5] dB** range, Gaussian soft targets (σ = 2 bins = 1 dB), BCE loss.
2. **Dry-energy-floor masking**: frames whose dry RMS < **−60 dBFS** are excluded from ALL loss terms — GR is a ratio of tiny RMS values in silence/fades and those labels are noise (the measured ±extremes were exactly these artifacts).
3. **Huber + weighted first-difference terms** on the differentiable softmax-expectation decode (dB domain): Huber regresses the curve directly and is robust to the transient spikes; the Δ-term penalises wrong attack/release slopes — the thing the conditioning knobs control.

`loss = BCE + 0.1·Huber(dB) + 1.0·Huber(ΔdB)` — components logged separately (`bce/`, `huber_db/`, `delta_db/`) for rebalancing.

Everything else identical: TFiLM (block 8 ≈ 46 ms), stateful TBPTT with both recurrent states carried per phase, 10 settings × 10 songs, same split/seed, cosine with committed budget.

In [ ]:
# ── 0. Dependencies ──────────────────────────────────────────────────
# nablafx goes in --no-deps so it can't clobber Colab's CUDA torch; its
# import chain needs the packages below. `rational` is only used by
# nablafx.processors.ddsp (which we never touch) and its wheel doesn't
# build on current Colab — stub it instead.
!pip install -q lightning torchmetrics soundfile auraloss einops wandb frechet-audio-distance
!pip install -q --no-deps nablafx

import sys, types

rational = types.ModuleType("rational")
rational.torch = types.ModuleType("rational.torch")
rational.torch.Rational = type("Rational", (), {})
sys.modules["rational"], sys.modules["rational.torch"] = rational, rational.torch

In [ ]:
# ── 1. Mount Drive (dataset) + clone repo from GitHub (code) ─────────
# The repo is NOT synced to Drive (only data/ is). Code comes from GitHub —
# push local changes before (re)running this cell; re-running pulls updates.

import os
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

DRIVE_DATA_ROOT = "/content/drive/Othercomputers/MacBook Air/data/Diff-SSL-G-Comp"
REPO_URL = "https://github.com/5aola/Virtual-Analogue-Compressor-Modelling.git"
REPO_ROOT = "/content/Virtual-Analogue-Compressor-Modelling"

if os.path.isdir(REPO_ROOT):
    !git -C "{REPO_ROOT}" pull --ff-only
else:
    !git clone --depth 1 "{REPO_URL}" "{REPO_ROOT}"

DATA_ROOT = DRIVE_DATA_ROOT
COND_DIR = os.path.join(REPO_ROOT, "05_conditioning")
OUTPUT_DIR = os.path.join(os.path.dirname(DATA_ROOT), "gr_pred_runs")

assert os.path.isdir(os.path.join(DATA_ROOT, "gr_curves")), f"Bad DATA_ROOT: {DATA_ROOT}"
assert os.path.isfile(os.path.join(COND_DIR, "dataset.py")), f"Clone failed: {REPO_ROOT}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for p in (REPO_ROOT, COND_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"REPO_ROOT  : {REPO_ROOT}")
print(f"DATA_ROOT  : {DATA_ROOT}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")

In [ ]:
# ── 2. Cache dataset to Colab local SSD ──────────────────────────────

import shutil
from dataset import discover_diffssl_gr_pairs

LOCAL_DATA_ROOT = "/content/Diff-SSL-G-Comp"

pairs = discover_diffssl_gr_pairs(DATA_ROOT)
settings = sorted({p["setting"] for p in pairs})
songs = sorted({p["song"] for p in pairs})
print(f"Caching {len(songs)} songs × {len(settings)} settings → {LOCAL_DATA_ROOT}")

local_dry = Path(LOCAL_DATA_ROOT) / "processed_normalized"
local_dry.mkdir(parents=True, exist_ok=True)
for song in songs:
    fn = f"{song}_UnmasteredWAV.wav"
    src = Path(DATA_ROOT) / "processed_normalized" / fn
    dst = local_dry / fn
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        shutil.copy2(src, dst)

for setting in settings:
    local_gr = Path(LOCAL_DATA_ROOT) / "gr_curves" / setting
    local_gr.mkdir(parents=True, exist_ok=True)
    for song in songs:
        fn = f"{song}.pt"
        src = Path(DATA_ROOT) / "gr_curves" / setting / fn
        if not src.is_file():
            continue
        dst = local_gr / fn
        if not dst.exists() or dst.stat().st_size != src.stat().st_size:
            shutil.copy2(src, dst)

DATA_ROOT = LOCAL_DATA_ROOT
print(f"Using local cache: {DATA_ROOT}")

In [ ]:
# ── 3. Imports & hyper-parameters ────────────────────────────────────

import json
from datetime import datetime

import torch
import lightning as pl
from lightning.pytorch.callbacks import (
    EarlyStopping, LearningRateMonitor, ModelCheckpoint, TQDMProgressBar,
)
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger

from dataset import (
    HOP_SIZE, SAMPLE_RATE, SEGMENT_LEN,
    MultiSettingGRDataModule, discover_diffssl_gr_pairs,
)
from model import StatefulTFiLMLSTMGRBins
from system import TFiLMGRBinsSystem
from splits import DIFFSSL_PARAM_RANGES, build_split_manifest
from gr_target import BIN_RESOLUTION_DB, GR_DB_MIN, GR_DB_MAX, NUM_BINS
from src.dsp import PARAM_ORDER
from src.dsp_torch import RMS_WINDOW

print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "WARNING: CPU runtime")

SPLIT_SEED = 42
N_VAL_SONGS = 1
N_TEST_SONGS = 2

LR = 1e-3
MAX_EPOCHS = 800
EARLY_STOP_PATIENCE = 200
WARMUP_FRAMES = 4
SCHEDULER = "cosine"
ETA_MIN = 1e-6

ENCODER_CHANNELS = 8
HIDDEN_SIZE = 24
TFILM_CHANNELS = 8
TFILM_BLOCK_SIZE = 8     # frames; 8 × 256 / 44100 ≈ 46 ms modulation rate
TFILM_NUM_LAYERS = 1

# loss (see system.TFiLMGRBinsSystem)
SIGMA_BINS = 2.0         # Gaussian soft-target width = 2 bins = 1.0 dB
ENERGY_FLOOR_DB = -60.0  # frames with dry RMS below this are masked from the loss
HUBER_WEIGHT = 0.1
HUBER_BETA_DB = 1.0      # quadratic below 1 dB error, linear above
DELTA_WEIGHT = 1.0       # weighted first-difference (attack/release timing)

print(f"Bins: {NUM_BINS} @ {BIN_RESOLUTION_DB:.2f} dB over [{GR_DB_MIN}, {GR_DB_MAX}] dB")

RUN_TAG = "lstm_tfilm_bins_maskedhuber_delta"
RESUME_RUN = None

In [ ]:
# ── 4. Preview split ───────────────────────────────────────────────

preview = build_split_manifest(
    discover_diffssl_gr_pairs(DATA_ROOT),
    seed=SPLIT_SEED,
    n_val_songs=N_VAL_SONGS,
    n_test_songs=N_TEST_SONGS,
)
print(f"Settings ({len(preview.all_settings)}): {preview.all_settings}")
print(f"Test settings (lowest T): {preview.test_settings}")
print(f"Train songs: {preview.train_songs}")
print(f"Val songs  : {preview.val_songs}")
print(f"Test songs : {preview.test_songs}")
print(
    f"Pairs — train={len(preview.train_pair_keys)} "
    f"val={len(preview.val_pair_keys)} test={len(preview.test_pair_keys)}"
)

In [ ]:
# ── 5. Model size ────────────────────────────────────────────────────

model = StatefulTFiLMLSTMGRBins(
    hop_size=HOP_SIZE,
    encoder_channels=ENCODER_CHANNELS,
    hidden_size=HIDDEN_SIZE,
    tfilm_channels=TFILM_CHANNELS,
    tfilm_block_size=TFILM_BLOCK_SIZE,
    tfilm_num_layers=TFILM_NUM_LAYERS,
    num_bins=NUM_BINS,
)
n_params = sum(p.numel() for p in model.parameters())
print(f"StatefulTFiLMLSTMGRBins: {n_params:,} params")
for name, mod in model.named_children():
    print(f"  {name:12s} {sum(p.numel() for p in mod.parameters()):,}")

In [ ]:
# ── 6. Train ─────────────────────────────────────────────────────────

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

assert DATA_ROOT.startswith("/content/"), "Run the cache cell first (cell 2)."

if RESUME_RUN:
    RUN_NAME = RESUME_RUN
    RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)
    _resume_ckpt = os.path.join(RUN_DIR, "checkpoints", "last.ckpt")
    print(f"RESUMING: {RUN_NAME}")
else:
    RUN_NAME = f"lstm_gr_{datetime.now():%Y%m%d_%H%M%S}_{RUN_TAG}"
    RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)
    _resume_ckpt = None
    print(f"NEW run: {RUN_NAME}")

os.makedirs(RUN_DIR, exist_ok=True)
split_path = os.path.join(RUN_DIR, "split_manifest.json")

dm = MultiSettingGRDataModule(
    data_root=DATA_ROOT,
    segment_len=SEGMENT_LEN,
    sample_rate=SAMPLE_RATE,
    split_seed=SPLIT_SEED,
    n_val_songs=N_VAL_SONGS,
    n_test_songs=N_TEST_SONGS,
    split_manifest_path=split_path,
)
dm.setup()  # idempotent — trainer.fit/.test reuse the cached datasets

print(f"Train/val/test streams: {dm.train_dataset.B} / {dm.val_dataset.B} / {dm.test_dataset.B}")
print(f"Steps/epoch (train): {len(dm.train_dataset)}")
print(f"Split manifest: {split_path}")

with open(os.path.join(RUN_DIR, "hparams.json"), "w") as f:
    json.dump({
        "approach": "gr_prediction_tfilm_conditioned_bins",
        "model_type": "stateful_tfilm_lstm_gr_bins",
        "dataset": "Diff-SSL-G-Comp",
        "setting": "multi (10 settings)",
        "conditioning": "nablafx.TFiLM",
        "sample_rate": SAMPLE_RATE,
        "hop_size": HOP_SIZE,
        "segment_len": SEGMENT_LEN,
        "rms_window": RMS_WINDOW,
        "gr_range_db": [GR_DB_MIN, GR_DB_MAX],
        "param_order": PARAM_ORDER,
        "param_ranges": DIFFSSL_PARAM_RANGES,
        "split_seed": SPLIT_SEED,
        "test_settings": dm.split.test_settings,
        "train_songs": dm.split.train_songs,
        "val_songs": dm.split.val_songs,
        "test_songs": dm.split.test_songs,
        "discretization": {
            "num_bins": NUM_BINS,
            "bin_resolution_db": BIN_RESOLUTION_DB,
            "sigma_bins": SIGMA_BINS,
        },
        "loss": {
            "kind": "masked_bce_huber_delta",
            "energy_floor_db": ENERGY_FLOOR_DB,
            "huber_weight": HUBER_WEIGHT,
            "huber_beta_db": HUBER_BETA_DB,
            "delta_weight": DELTA_WEIGHT,
            "warmup_frames": WARMUP_FRAMES,
        },
        "model": {
            "encoder_channels": ENCODER_CHANNELS,
            "hidden_size": HIDDEN_SIZE,
            "tfilm_channels": TFILM_CHANNELS,
            "tfilm_block_size": TFILM_BLOCK_SIZE,
            "tfilm_num_layers": TFILM_NUM_LAYERS,
            "num_bins": NUM_BINS,
            "num_params": n_params,
        },
        "lr": LR,
        "max_epochs": MAX_EPOCHS,
        "early_stop_patience": EARLY_STOP_PATIENCE,
        "scheduler": SCHEDULER,
        "eta_min": ETA_MIN,
    }, f, indent=2)

system = TFiLMGRBinsSystem(
    model=model,
    lr=LR,
    warmup_frames=WARMUP_FRAMES,
    sigma_bins=SIGMA_BINS,
    energy_floor_db=ENERGY_FLOOR_DB,
    huber_weight=HUBER_WEIGHT,
    huber_beta_db=HUBER_BETA_DB,
    delta_weight=DELTA_WEIGHT,
    scheduler=SCHEDULER,
    max_epochs=MAX_EPOCHS,
    eta_min=ETA_MIN,
)


class ResumeOverrides(pl.Callback):
    """Checkpoint restore overwrites cosine T_max and EarlyStopping's
    patience / wait_count / best_score with the old run's values — re-apply
    the notebook hyper-parameters after restore so an extended budget and a
    fresh early-stop window actually take effect."""

    def on_train_start(self, trainer, pl_module):
        sched = trainer.lr_scheduler_configs[0].scheduler
        if hasattr(sched, "T_max"):
            sched.T_max = MAX_EPOCHS
        for cb in trainer.callbacks:
            if isinstance(cb, EarlyStopping):
                cb.patience = EARLY_STOP_PATIENCE
                cb.wait_count = 0
                cb.best_score = torch.tensor(float("inf"))


ckpt_dir = os.path.join(RUN_DIR, "checkpoints")
callbacks = [
    ModelCheckpoint(
        dirpath=ckpt_dir,
        monitor="loss/val",
        mode="min",
        save_top_k=3,
        save_last=True,
        filename="best-{epoch:03d}-{step}",
        auto_insert_metric_name=False,
    ),
    LearningRateMonitor(logging_interval="epoch"),
    EarlyStopping(monitor="loss/val", mode="min", patience=EARLY_STOP_PATIENCE, verbose=True),
    TQDMProgressBar(refresh_rate=10),
    ResumeOverrides(),
]
loggers = [
    TensorBoardLogger(save_dir=RUN_DIR, name="tb", version=""),
    CSVLogger(save_dir=RUN_DIR, name="csv", version=""),
]

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator="gpu",
    devices=1,
    callbacks=callbacks,
    logger=loggers,
    gradient_clip_val=1.0,
    log_every_n_steps=10,
)

trainer.fit(system, dm, ckpt_path=_resume_ckpt)

In [ ]:
# ── 7. Test (held-out songs × lowest-threshold settings) ─────────────

trainer.test(system, datamodule=dm, ckpt_path=callbacks[0].best_model_path)

In [ ]:
# ── 8. Plot: predicted vs target GR on val streams ───────────────────
# Streams the val set in order so both recurrent states settle, then
# plots a mid-track chunk per stream.

import matplotlib.pyplot as plt
import numpy as np

best = torch.load(callbacks[0].best_model_path, map_location="cuda", weights_only=False)
system.load_state_dict(best["state_dict"])
system.eval().cuda()
print(f"Loaded best checkpoint: {callbacks[0].best_model_path}")

val_steps = list(dm.val_dataloader())
pick = len(val_steps) // 2

state = None
with torch.no_grad():
    for s, (dry, gr_db, params, mask, reset) in enumerate(val_steps):
        if bool(reset):
            state = None
            system.model.tfilm.hidden_state = None
        logits, state = system.model(dry.cuda(), params.cuda(), state, return_state=True)
        if s == pick:
            pred_db = system.model.to_db(logits, sample_len=dry.shape[-1]).cpu().numpy()
            target_db = gr_db.numpy()
            rows = torch.nonzero(mask).squeeze(1).tolist()
            break

n_plots = min(4, len(rows))
fig, axes = plt.subplots(n_plots, 1, figsize=(14, 3 * n_plots), sharex=True, squeeze=False)
t = np.arange(pred_db.shape[-1]) / SAMPLE_RATE
for ax, r in zip(axes[:, 0], rows[:n_plots]):
    ax.plot(t, target_db[r, 0], label="Target GR", alpha=0.8, linewidth=0.5)
    ax.plot(t, pred_db[r, 0], label="Predicted GR", alpha=0.8, linewidth=0.5)
    l1 = float(np.mean(np.abs(pred_db[r, 0] - target_db[r, 0])))
    c = dm.val_dataset.cache[r]
    ax.set_title(f"{c['song']} / {c['setting']} (chunk {pick}) — L1 = {l1:.2f} dB")
    ax.set_ylabel("GR (dB)")
    ax.legend(loc="lower right", fontsize=8)
axes[-1, 0].set_xlabel("Time (s)")
fig.tight_layout()
plot_path = os.path.join(RUN_DIR, "eval_gr_comparison.png")
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"Saved plot -> {plot_path}")
plt.show()

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{RUN_DIR}/tb"